In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/mental-health-trials-pipeline'
for sub in ['data/raw', 'data/clean', 'sql', 'results']:
    os.makedirs(f'{PROJECT}/{sub}', exist_ok=True)
print(os.listdir(PROJECT))

Mounted at /content/drive
['data', 'sql', 'results']


In [ ]:
import requests, json, time

BASE = "https://clinicaltrials.gov/api/v2/studies"
test = requests.get(BASE, params={
    "query.cond": "depression",
    "pageSize": 1,
    "countTotal": "true",
}, timeout=60)

print("Status:", test.status_code)
data = test.json()
print("Total depression studies:", data.get("totalCount"))
print("Sections in one study:", list(data["studies"][0]["protocolSection"].keys()))

Status: 200
Total depression studies: 12834
Sections in one study: ['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'oversightModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule', 'ipdSharingStatementModule']


In [ ]:
params = {
    "query.cond": ('depression OR anxiety OR PTSD OR "bipolar disorder" '
                   'OR schizophrenia OR psychosis OR "substance use disorder"'),
    "filter.advanced": ("AREA[StudyType]INTERVENTIONAL AND "
                        "AREA[StartDate]RANGE[2010-01-01,2025-12-31]"),
    "pageSize": 1000,
    "countTotal": "true",
    "format": "json",
}

studies, token = [], None
while True:
    if token:
        params["pageToken"] = token
    r = requests.get(BASE, params=params, timeout=120)
    r.raise_for_status()
    page = r.json()
    if "totalCount" in page:
        print("Studies matching the search:", page["totalCount"])
    studies += page["studies"]
    token = page.get("nextPageToken")
    print(len(studies), "downloaded")
    if not token:
        break
    time.sleep(0.3)   # be polite to the server

with open(f"{PROJECT}/data/raw/trials_raw.json", "w") as f:
    json.dump(studies, f)
print("Saved", len(studies), "studies")

Studies matching the search: 17581
1000 downloaded
2000 downloaded
3000 downloaded
4000 downloaded
5000 downloaded
6000 downloaded
7000 downloaded
8000 downloaded
9000 downloaded
10000 downloaded
11000 downloaded
12000 downloaded
13000 downloaded
14000 downloaded
15000 downloaded
16000 downloaded
17000 downloaded
17581 downloaded
Saved 17581 studies


In [ ]:
import pandas as pd

def g(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d

with open(f"{PROJECT}/data/raw/trials_raw.json") as f:
    studies = json.load(f)

trials, conds, ints, sites = [], [], [], []
for s in studies:
    p = s.get("protocolSection", {})
    nct = g(p, "identificationModule", "nctId")
    start = g(p, "statusModule", "startDateStruct", "date")
    trials.append({
        "nct_id": nct,
        "title": g(p, "identificationModule", "briefTitle"),
        "start_date": start,
        "start_year": int(start[:4]) if start else None,
        "overall_status": g(p, "statusModule", "overallStatus"),
        "why_stopped": g(p, "statusModule", "whyStopped"),
        "phase": "/".join(g(p, "designModule", "phases", default=[]) or []) or "NA",
        "enrollment": g(p, "designModule", "enrollmentInfo", "count"),
        "sponsor_name": g(p, "sponsorCollaboratorsModule", "leadSponsor", "name"),
        "sponsor_class": g(p, "sponsorCollaboratorsModule", "leadSponsor", "class"),
    })
    for c in g(p, "conditionsModule", "conditions", default=[]) or []:
        conds.append({"nct_id": nct, "condition": c})
    for i in g(p, "armsInterventionsModule", "interventions", default=[]) or []:
        ints.append({"nct_id": nct,
                     "intervention_type": i.get("type"),
                     "intervention_name": i.get("name")})
    locations = g(p, "contactsLocationsModule", "locations", default=[]) or []
    for country in {loc.get("country") for loc in locations if loc.get("country")}:
        sites.append({"nct_id": nct, "country": country})

trials = pd.DataFrame(trials)
conditions = pd.DataFrame(conds)
interventions = pd.DataFrame(ints)
sites = pd.DataFrame(sites)

for name, df in [("trials", trials), ("conditions", conditions),
                 ("interventions", interventions), ("sites", sites)]:
    print(f"{name:14s} {len(df):>8,} rows")
print(trials.head())

trials           17,581 rows
conditions       39,080 rows
interventions    31,287 rows
sites            19,280 rows
        nct_id                                              title  start_date  \
0  NCT02158429  Study of Ultrasound To Increase Parental Bondi...     2011-04   
1  NCT04131829  Brain Network Changes Accompanying and Predict...  2019-10-14   
2  NCT04587531           Neural Mechanisms With CES in Depression  2021-07-05   
3  NCT05662137  The Effect of Recognition and Expression of Em...  2023-01-15   
4  NCT04692272  Xbox Kinect, Sleep Quality, Anxiety and Functi...  2014-12-15   

   start_year overall_status        why_stopped          phase  enrollment  \
0        2011        UNKNOWN               None             NA       120.0   
1        2019     RECRUITING               None  PHASE1/PHASE2       100.0   
2        2021     TERMINATED  Lack of financing             NA         1.0   
3        2023      COMPLETED               None             NA        36.0   
4      

In [ ]:
print("IDs unique:", trials.nct_id.is_unique)
print("Start years:", trials.start_year.min(), "to", trials.start_year.max())
print(trials.overall_status.value_counts())
print(trials.phase.value_counts().head(8))

IDs unique: True
Start years: 2010 to 2025
overall_status
COMPLETED                  10484
UNKNOWN                     2575
RECRUITING                  1954
TERMINATED                   849
ACTIVE_NOT_RECRUITING        595
WITHDRAWN                    497
NOT_YET_RECRUITING           370
ENROLLING_BY_INVITATION      191
SUSPENDED                     66
Name: count, dtype: int64
phase
NA               13533
PHASE2            1172
PHASE4             874
PHASE3             756
PHASE1             624
PHASE1/PHASE2      225
EARLY_PHASE1       200
PHASE2/PHASE3      197
Name: count, dtype: int64


In [ ]:
params = {
    "query.cond": ('depression OR anxiety OR PTSD OR "bipolar disorder" '
               'OR schizophrenia OR psychosis OR "substance use disorder" '
               'OR "obsessive-compulsive disorder" OR "eating disorder" '
               'OR "anorexia nervosa" OR "bulimia nervosa" OR "binge eating disorder"'),
    "filter.advanced": ("AREA[StudyType]INTERVENTIONAL AND "
                        "AREA[StartDate]RANGE[2010-01-01,2025-12-31]"),
    "pageSize": 1000,
    "countTotal": "true",
    "format": "json",
}

studies, token = [], None
while True:
    if token:
        params["pageToken"] = token
    r = requests.get(BASE, params=params, timeout=120)
    r.raise_for_status()
    page = r.json()
    if "totalCount" in page:
        print("Studies matching the search:", page["totalCount"])
    studies += page["studies"]
    token = page.get("nextPageToken")
    print(len(studies), "downloaded")
    if not token:
        break
    time.sleep(0.3)   # be polite to the server

with open(f"{PROJECT}/data/raw/trials_raw.json", "w") as f:
    json.dump(studies, f)
print("Saved", len(studies), "studies")

Studies matching the search: 18357
1000 downloaded
2000 downloaded
3000 downloaded
4000 downloaded
5000 downloaded
6000 downloaded
7000 downloaded
8000 downloaded
9000 downloaded
10000 downloaded
11000 downloaded
12000 downloaded
13000 downloaded
14000 downloaded
15000 downloaded
16000 downloaded
17000 downloaded
18000 downloaded
18357 downloaded
Saved 18357 studies


In [ ]:
print("IDs unique:", trials.nct_id.is_unique)
print("Start years:", trials.start_year.min(), "to", trials.start_year.max())
print(trials.overall_status.value_counts())
print(trials.phase.value_counts().head(8))

IDs unique: True
Start years: 2010 to 2025
overall_status
COMPLETED                  10484
UNKNOWN                     2575
RECRUITING                  1954
TERMINATED                   849
ACTIVE_NOT_RECRUITING        595
WITHDRAWN                    497
NOT_YET_RECRUITING           370
ENROLLING_BY_INVITATION      191
SUSPENDED                     66
Name: count, dtype: int64
phase
NA               13533
PHASE2            1172
PHASE4             874
PHASE3             756
PHASE1             624
PHASE1/PHASE2      225
EARLY_PHASE1       200
PHASE2/PHASE3      197
Name: count, dtype: int64


In [ ]:
import pandas as pd

def g(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d

with open(f"{PROJECT}/data/raw/trials_raw.json") as f:
    studies = json.load(f)

trials, conds, ints, sites = [], [], [], []
for s in studies:
    p = s.get("protocolSection", {})
    nct = g(p, "identificationModule", "nctId")
    start = g(p, "statusModule", "startDateStruct", "date")
    trials.append({
        "nct_id": nct,
        "title": g(p, "identificationModule", "briefTitle"),
        "start_date": start,
        "start_year": int(start[:4]) if start else None,
        "overall_status": g(p, "statusModule", "overallStatus"),
        "why_stopped": g(p, "statusModule", "whyStopped"),
        "phase": "/".join(g(p, "designModule", "phases", default=[]) or []) or "NA",
        "enrollment": g(p, "designModule", "enrollmentInfo", "count"),
        "sponsor_name": g(p, "sponsorCollaboratorsModule", "leadSponsor", "name"),
        "sponsor_class": g(p, "sponsorCollaboratorsModule", "leadSponsor", "class"),
    })
    for c in g(p, "conditionsModule", "conditions", default=[]) or []:
        conds.append({"nct_id": nct, "condition": c})
    for i in g(p, "armsInterventionsModule", "interventions", default=[]) or []:
        ints.append({"nct_id": nct,
                     "intervention_type": i.get("type"),
                     "intervention_name": i.get("name")})
    locations = g(p, "contactsLocationsModule", "locations", default=[]) or []
    for country in {loc.get("country") for loc in locations if loc.get("country")}:
        sites.append({"nct_id": nct, "country": country})

trials = pd.DataFrame(trials)
conditions = pd.DataFrame(conds)
interventions = pd.DataFrame(ints)
sites = pd.DataFrame(sites)

for name, df in [("trials", trials), ("conditions", conditions),
                 ("interventions", interventions), ("sites", sites)]:
    print(f"{name:14s} {len(df):>8,} rows")
print(trials.head())

trials           18,357 rows
conditions       40,636 rows
interventions    32,628 rows
sites            20,028 rows
        nct_id                                              title  start_date  \
0  NCT02158429  Study of Ultrasound To Increase Parental Bondi...     2011-04   
1  NCT04131829  Brain Network Changes Accompanying and Predict...  2019-10-14   
2  NCT04587531           Neural Mechanisms With CES in Depression  2021-07-05   
3  NCT05662137  The Effect of Recognition and Expression of Em...  2023-01-15   
4  NCT04692272  Xbox Kinect, Sleep Quality, Anxiety and Functi...  2014-12-15   

   start_year overall_status        why_stopped          phase  enrollment  \
0        2011        UNKNOWN               None             NA       120.0   
1        2019     RECRUITING               None  PHASE1/PHASE2       100.0   
2        2021     TERMINATED  Lack of financing             NA         1.0   
3        2023      COMPLETED               None             NA        36.0   
4      

In [ ]:
import re

# Checked first: conditions that look like mental health but aren't
EXCLUDE = (r"respiratory depression|cns depression|preoperative anxiety"
           r"|pre-operative anxiety|dental anxiety|procedural anxiety"
           r"|traumatic brain injury")

GROUPS = {
    "Depression": r"depress|dysthym|\bmdd\b|anhedonia",
    "Anxiety": r"anxi|panic|phobi|\bgad\b|agoraphobi",
    "OCD": r"obsessive|compulsive|\bocd\b",
    "PTSD": r"ptsd|post-?traumatic|posttraumatic|acute stress disorder",
    "Bipolar disorder": r"bipolar|\bmania\b|manic",
    "Schizophrenia & psychosis": r"schizo|psychos|psychotic",
    "Substance use": (r"substance|alcohol|opioid|opiate|cocaine|methamphetamine"
                      r"|stimulant use|cannabis use|addict|use disorder|dependence"),
    "Eating disorders": r"anorexia nervosa|bulimi|binge.?eating|eating disorder|\barfid\b",
}

def group_of(text):
    t = (text or "").lower()
    if re.search(EXCLUDE, t):
        return "Other"
    for name, pattern in GROUPS.items():
        if re.search(pattern, t):
            return name
    return "Other"

conditions["condition_group"] = conditions["condition"].map(group_of)
print(conditions["condition_group"].value_counts())
print(conditions.loc[conditions.condition_group == "Other", "condition"]
      .value_counts().head(30))

# Spot-check: random examples from each group, to catch wrong matches
for grp in GROUPS:
    print(f"\n--- {grp} ---")
    print(conditions.loc[conditions.condition_group == grp, "condition"]
          .drop_duplicates().sample(8, random_state=1).tolist())

condition_group
Other                        19949
Depression                    7893
Anxiety                       4768
Schizophrenia & psychosis     3065
PTSD                          1467
Substance use                 1364
Eating disorders               891
Bipolar disorder               783
OCD                            456
Name: count, dtype: int64
condition
Pain                              432
Stress                            429
Post Traumatic Stress Disorder    308
Quality of Life                   219
Insomnia                          174
Dental Anxiety                    167
Mental Disorder                   146
Obesity                           137
Virtual Reality                   120
Cancer                            119
Suicidal Ideation                 118
Mood Disorders                    109
Chronic Pain                      108
Psychological Distress            103
Sleep                              96
Cognitive Impairment               93
Mental Health            

In [ ]:
PSYCHEDELIC = (
    # substances
    r"psilocybin|psilocin|mdma|midomafetamine|\blsd\b|lysergic"
    r"|\bdmt\b|dimethyltryptamine|5-meo|ayahuasca|ibogaine|noribogaine"
    r"|mescaline|salvinorin|psychedelic"
    # company drug codes
    r"|comp ?360"        # Compass Pathways, psilocybin
    r"|mm-?120"          # MindMed, LSD
    r"|cyb00[34]"        # Cybin, deuterated psilocybin analogue / DMT
    r"|gh00[12]"         # GH Research, 5-MeO-DMT
    r"|bpl-?003"         # Beckley Psytech, 5-MeO-DMT nasal spray
    r"|spl0?26"          # Small Pharma, DMT
    r"|re-?104"          # Reunion Neuroscience, psilocybin-like prodrug
)
KETAMINE = r"ketamine|spravato"   # ketamine also matches esketamine and arketamine

def drug_class(name):
    n = (name or "").lower()
    if n.startswith("placebo") or "matched placebo" in n:
        return None
    if re.search(PSYCHEDELIC, n):
        return "Psychedelic / MDMA"
    if re.search(KETAMINE, n):
        return "Ketamine / esketamine"
    return None

interventions["psychedelic_class"] = interventions["intervention_name"].map(drug_class)
print(interventions["psychedelic_class"].value_counts())
print(interventions.loc[interventions.psychedelic_class.notna(),
                        "intervention_name"].value_counts().head(40))

psychedelic_class
Ketamine / esketamine    415
Psychedelic / MDMA       202
Name: count, dtype: int64
intervention_name
Ketamine                                           165
Psilocybin                                          67
Esketamine                                          40
Ketamine Hydrochloride                              30
MDMA                                                13
ketamine                                            13
Esketamine 56 mg                                     6
Esketamine 84 mg                                     6
Midomafetamine                                       6
Psilocybin 25 mg                                     6
Midomafetamine HCl                                   6
Psilocybin therapy                                   5
S-ketamine                                           5
Esketamine hydrochloride                             4
Intravenous Ketamine                                 4
LSD                                                  4


In [ ]:
psy_out = flagged[(flagged.psychedelic_class == "Psychedelic / MDMA")
                  & ~flagged.nct_id.isin(grouped_ids)].nct_id.unique()
check = list(psy_out) + ["NCT06498700"]
print(conditions[conditions.nct_id.isin(check)]
      .sort_values("nct_id")[["nct_id", "condition", "condition_group"]]
      .to_string())

            nct_id                                    condition condition_group
38726  NCT02308969                                      Healthy           Other
34946  NCT04842045                      Psychedelic Experiences           Other
34947  NCT04842045                                      Amnesia           Other
22790  NCT05847686  Hematopoietic and Lymphatic System Neoplasm           Other
22791  NCT05847686          Metastatic Malignant Solid Neoplasm           Other
9913   NCT06498700        Subanesthetic Ketamine Dexmedetomidin           Other
37969  NCT06644170  Hematopoietic and Lymphatic System Neoplasm           Other
37970  NCT06644170          Metastatic Malignant Solid Neoplasm           Other
274    NCT06816667                           Healthy Volunteers           Other
34590  NCT07002034                          Adjustment Disorder           Other
27277  NCT07710027                             Healthy Controls           Other


In [ ]:
trials["in_scope"] = trials["nct_id"].isin(grouped_ids).astype(int)
print(trials["in_scope"].value_counts())

in_scope
1    14792
0     3565
Name: count, dtype: int64


In [ ]:
PSYCHEDELIC = (
    # substances
    r"psilocybin|psilocin|mdma|midomafetamine|\blsd\b|lysergic"
    r"|\bdmt\b|dimethyltryptamine|5-meo|ayahuasca|ibogaine|noribogaine"
    r"|mescaline|salvinorin|psychedelic"
    # company drug codes

    r"|comp ?360"        # Compass Pathways, psilocybin
    r"|mm-?120"          # MindMed, LSD
    r"|cyb00[34]"        # Cybin, deuterated psilocybin analogue / DMT
    r"|gh00[12]"         # GH Research, 5-MeO-DMT
    r"|bpl-?003"         # Beckley Psytech, 5-MeO-DMT nasal spray
    r"|spl0?26"          # Small Pharma, DMT
    r"|re-?104"          # Reunion Neuroscience, psilocybin-like prodrug
    r"|bmnd0\d"          # Biomind Labs, 5-MeO-DMT
)
KETAMINE = r"ketamine|spravato"   # ketamine also matches esketamine and arketamine

def drug_class(name):
    n = (name or "").lower()
    if n.startswith("placebo") or "matched placebo" in n:
        return None
    if re.search(PSYCHEDELIC, n):
        return "Psychedelic / MDMA"
    if re.search(KETAMINE, n):
        return "Ketamine / esketamine"
    return None

interventions["psychedelic_class"] = interventions["intervention_name"].map(drug_class)
print(interventions["psychedelic_class"].value_counts())
print(interventions.loc[interventions.psychedelic_class.notna(),
                        "intervention_name"].value_counts().head(40))

psychedelic_class
Ketamine / esketamine    415
Psychedelic / MDMA       203
Name: count, dtype: int64
intervention_name
Ketamine                                           165
Psilocybin                                          67
Esketamine                                          40
Ketamine Hydrochloride                              30
ketamine                                            13
MDMA                                                13
Esketamine 84 mg                                     6
Esketamine 56 mg                                     6
Midomafetamine                                       6
Midomafetamine HCl                                   6
Psilocybin 25 mg                                     6
S-ketamine                                           5
Psilocybin therapy                                   5
Esketamine hydrochloride                             4
LSD                                                  4
Intravenous Ketamine                                 4


In [ ]:
trials["in_scope"] = trials["nct_id"].isin(grouped_ids).astype(int)
print(trials["in_scope"].value_counts())

in_scope
1    14792
0     3565
Name: count, dtype: int64


In [ ]:
print(trials.columns.tolist())

['nct_id', 'title', 'start_date', 'start_year', 'overall_status', 'why_stopped', 'phase', 'enrollment', 'sponsor_name', 'sponsor_class', 'in_scope']


In [ ]:
d1 = q("""
SELECT t.start_year, c.condition_group,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t
JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY t.start_year, c.condition_group
ORDER BY t.start_year, c.condition_group;
""")
print(d1.pivot(index="start_year", columns="condition_group", values="trials").fillna(0).astype(int).to_string())

condition_group  Anxiety  Bipolar disorder  Depression  Eating disorders  OCD  PTSD  Schizophrenia & psychosis  Substance use
start_year                                                                                                                   
2010                  80                50         265                17   11    71                        127             23
2011                  85                40         267                15   20    62                        136             26
2012                 103                40         273                24   20    86                        136             32
2013                  99                42         244                20   19    74                        128             35
2014                 141                54         304                24   29    67                        155             29
2015                 163                36         326                36   23    77                        125        

In [ ]:
print(q("""
SELECT c.condition, COUNT(DISTINCT t.nct_id) AS trials
FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group = 'Anxiety' AND t.start_year = 2021
GROUP BY c.condition ORDER BY trials DESC LIMIT 20;
""").to_string())

                         condition  trials
0                          Anxiety     260
1                Anxiety Disorders      39
2     Generalized Anxiety Disorder      17
3          Social Anxiety Disorder      15
4                    Anxiety State      12
5                 Anxiety and Fear       9
6                   Panic Disorder       8
7   Anxiety Disorders and Symptoms       8
8                   Social Anxiety       7
9                 Anxiety Symptoms       6
10              Postpartum Anxiety       4
11           Postoperative Anxiety       4
12               Perinatal Anxiety       4
13                   Kinesiophobia       4
14                    Test Anxiety       3
15                    Fear Anxiety       3
16           Anxiety Postoperative       3
17                Anxiety Disorder       3
18                     Agoraphobia       3
19                 Specific Phobia       2


In [ ]:
flagged = interventions.dropna(subset=["psychedelic_class"])

# 1. What else do the "Psychedelic Effects" trials list?
ids = flagged.loc[flagged.intervention_name.str.contains("psychedelic effects", case=False), "nct_id"]
print(interventions[interventions.nct_id.isin(ids)][["nct_id", "intervention_type", "intervention_name"]])

# 2 and 3. Distinct trials per class, split by whether the trial has at least one grouped condition
grouped_ids = set(conditions.loc[conditions.condition_group != "Other", "nct_id"])
summary = (flagged.assign(in_scope=flagged.nct_id.isin(grouped_ids))
                  .groupby(["psychedelic_class", "in_scope"])["nct_id"].nunique()
                  .unstack(fill_value=0))
print(summary)

# A sample of the out-of-scope ketamine trials, to see what they are
out = flagged[~flagged.nct_id.isin(grouped_ids)].nct_id.unique()
print(trials[trials.nct_id.isin(out)][["nct_id", "title"]].head(15).to_string())

           nct_id intervention_type  \
266   NCT06816667              DRUG   
267   NCT06816667         PROCEDURE   
268   NCT06816667   DIAGNOSTIC_TEST   
269   NCT06816667   DIAGNOSTIC_TEST   
270   NCT06816667   DIAGNOSTIC_TEST   
271   NCT06816667   DIAGNOSTIC_TEST   
272   NCT06816667   DIAGNOSTIC_TEST   
273   NCT06816667   DIAGNOSTIC_TEST   
7752  NCT06812221              DRUG   
7753  NCT06812221         PROCEDURE   
7754  NCT06812221   DIAGNOSTIC_TEST   
7755  NCT06812221   DIAGNOSTIC_TEST   
7756  NCT06812221   DIAGNOSTIC_TEST   
7757  NCT06812221   DIAGNOSTIC_TEST   
7758  NCT06812221   DIAGNOSTIC_TEST   

                                      intervention_name  
266                                              BMND08  
267                   Brain electrical activity monitor  
268   Pharmacokinetic Study of Sublingually Administ...  
269                          Biochemical determinations  
270     Acute Subjective Ratings of Psychedelic Effects  
271                        

In [ ]:
import re

# Checked first: conditions that look like mental health but aren't
EXCLUDE = (r"respiratory depression|cns depression"
           r"|(pre|post|peri)[- ]?operative anxiety|anxiety,? (pre|post|peri)[- ]?operative"
           r"|presurgical|pre-surgical|surgical anxiety|surgery anxiety"
           r"|dental anxiety|dental fear|procedural anxiety|kinesiophobia|traumatic brain injury")
GROUPS = {
    "Depression": r"depress|dysthym|\bmdd\b|anhedonia",
    "Anxiety": r"anxi|panic|phobi|\bgad\b|agoraphobi",
    "OCD": r"obsessive|\bocd\b",
    "PTSD": r"ptsd|post[- ]?traumatic|acute stress disorder",
    "Bipolar disorder": r"bipolar|\bmania\b|manic",
    "Schizophrenia & psychosis": r"schizo|psychosis|psychoses|psychotic",
    "Substance use": (r"substance|alcohol use|alcohol abuse|alcohol dependence|alcoholism"
                      r"|hazardous alcohol|heavy drinking|binge drinking"
                      r"|opioid|opiate|cocaine|methamphetamine|stimulant use|cannabis use"
                      r"|addict|use disorder|dependence"),
    "Eating disorders": r"anorexia nervosa|bulimi|binge.?eating|eating disorder|\barfid\b",
}

def group_of(text):
    t = (text or "").lower()
    if re.search(EXCLUDE, t):
        return "Other"
    for name, pattern in GROUPS.items():
        if re.search(pattern, t):
            return name
    return "Other"

conditions["condition_group"] = conditions["condition"].map(group_of)
print(conditions["condition_group"].value_counts())
print(conditions.loc[conditions.condition_group == "Other", "condition"]
      .value_counts().head(30))

# Spot-check: random examples from each group, to catch wrong matches
for grp in GROUPS:
    print(f"\n--- {grp} ---")
    print(conditions.loc[conditions.condition_group == grp, "condition"]
          .drop_duplicates().sample(8, random_state=1).tolist())

condition_group
Other                        19830
Depression                    7893
Anxiety                       4661
Schizophrenia & psychosis     3021
PTSD                          1801
Substance use                 1307
Eating disorders               891
Bipolar disorder               783
OCD                            449
Name: count, dtype: int64
condition
Pain                        432
Stress                      429
Quality of Life             219
Insomnia                    174
Dental Anxiety              167
Mental Disorder             146
Obesity                     137
Virtual Reality             120
Cancer                      119
Suicidal Ideation           118
Mood Disorders              109
Chronic Pain                108
Psychological Distress      103
Sleep                        96
Cognitive Impairment         93
Mental Health                87
Fear                         87
Breast Cancer                87
Healthy                      85
Stroke                   

In [ ]:
PSYCHEDELIC = (
    # substances
    r"psilocybin|psilocin|mdma|midomafetamine|\blsd\b|lysergic"
    r"|\bdmt\b|dimethyltryptamine|5-meo|ayahuasca|ibogaine|noribogaine"
    r"|mescaline|salvinorin|psychedelic"
    # company drug codes
    r"|comp ?360"        # Compass Pathways, psilocybin
    r"|mm-?120"          # MindMed, LSD
    r"|cyb00[34]"        # Cybin, deuterated psilocybin analogue / DMT
    r"|gh00[12]"         # GH Research, 5-MeO-DMT
    r"|bpl-?003"         # Beckley Psytech, 5-MeO-DMT nasal spray
    r"|spl0?26"          # Small Pharma, DMT
    r"|re-?104"          # Reunion Neuroscience, psilocybin-like prodrug
    r"|bmnd0\d"          # Biomind Labs, 5-MeO-DMT
)
KETAMINE = r"ketamine|spravato"   # ketamine also matches esketamine and arketamine

def drug_class(name):
    n = (name or "").lower()
    if n.startswith("placebo") or "matched placebo" in n:
        return None
    if re.search(PSYCHEDELIC, n):
        return "Psychedelic / MDMA"
    if re.search(KETAMINE, n):
        return "Ketamine / esketamine"
    return None

interventions["psychedelic_class"] = interventions["intervention_name"].map(drug_class)
print(interventions["psychedelic_class"].value_counts())
print(interventions.loc[interventions.psychedelic_class.notna(),
                        "intervention_name"].value_counts().head(40))

psychedelic_class
Ketamine / esketamine    415
Psychedelic / MDMA       203
Name: count, dtype: int64
intervention_name
Ketamine                                           165
Psilocybin                                          67
Esketamine                                          40
Ketamine Hydrochloride                              30
ketamine                                            13
MDMA                                                13
Esketamine 84 mg                                     6
Esketamine 56 mg                                     6
Midomafetamine                                       6
Midomafetamine HCl                                   6
Psilocybin 25 mg                                     6
S-ketamine                                           5
Psilocybin therapy                                   5
Esketamine hydrochloride                             4
LSD                                                  4
Intravenous Ketamine                                 4


In [ ]:
trials["in_scope"] = trials["nct_id"].isin(grouped_ids).astype(int)
print(trials["in_scope"].value_counts())

in_scope
1    14792
0     3565
Name: count, dtype: int64


In [ ]:
import sqlite3

tables = {"trials": trials, "conditions": conditions,
          "interventions": interventions, "sites": sites}
for name, df in tables.items():
    df.to_csv(f"{PROJECT}/data/clean/{name}.csv", index=False)

con = sqlite3.connect(f"{PROJECT}/data/trials.db")
for name, df in tables.items():
    df.to_sql(name, con, if_exists="replace", index=False)

def q(sql):
    return pd.read_sql(sql, con)

print(q("SELECT name FROM sqlite_master WHERE type = 'table'"))

            name
0         trials
1     conditions
2  interventions
3          sites


In [ ]:
d1 = q("""
SELECT t.start_year, c.condition_group,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t
JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY t.start_year, c.condition_group
ORDER BY t.start_year, c.condition_group;
""")
print(d1.pivot(index="start_year", columns="condition_group", values="trials").fillna(0).astype(int).to_string())

condition_group  Anxiety  Bipolar disorder  Depression  Eating disorders  OCD  PTSD  Schizophrenia & psychosis  Substance use
start_year                                                                                                                   
2010                  80                50         265                17   11    71                        127             23
2011                  85                40         267                15   20    62                        136             26
2012                 102                40         273                24   20    86                        136             32
2013                  99                42         244                20   19    74                        128             35
2014                 141                54         304                24   29    67                        155             29
2015                 163                36         326                36   23    77                        125        

In [ ]:
print(q("""
SELECT t.start_year,
       COUNT(DISTINCT t.nct_id) AS anxiety_trials,
       COUNT(DISTINCT CASE WHEN LOWER(t.title) LIKE '%covid%'
                            OR LOWER(t.title) LIKE '%pandemic%'
                            OR LOWER(t.title) LIKE '%coronavirus%'
                            OR LOWER(t.title) LIKE '%sars-cov%'
                           THEN t.nct_id END) AS covid_in_title
FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group = 'Anxiety' AND t.start_year BETWEEN 2018 AND 2023
GROUP BY t.start_year ORDER BY t.start_year;
""").to_string())

   start_year  anxiety_trials  covid_in_title
0        2018             239               0
1        2019             258               0
2        2020             263              38
3        2021             415              25
4        2022             392               7
5        2023             477               7


In [60]:
import re

EXCLUDE = (r"respiratory depression|cns depression"
           r"|(pre|post|peri)[- ]?operative anxiety|anxiety,? (pre|post|peri)[- ]?operative"
           r"|presurgical|pre-surgical|surgical anxiety|surgery anxiety"
           r"|dental anxiety|dental fear|procedural anxiety|kinesiophobia|traumatic brain injury")

GROUPS = {
    "Depression": r"depress|dysthym|\bmdd\b|anhedonia",
    "Anxiety": r"anxi|panic|phobi|\bgad\b|agoraphobi",
    "OCD": r"obsessive|\bocd\b",
    "PTSD": r"ptsd|post[- ]?traumatic|acute stress disorder",
    "Bipolar disorder": r"bipolar|\bmania\b|manic",
    "Schizophrenia & psychosis": r"schizo|psychosis|psychoses|psychotic",
    "Substance use": (r"substance|alcohol use|alcohol abuse|alcohol dependence|alcoholism"
                      r"|hazardous alcohol|heavy drinking|binge drinking"
                      r"|opioid|opiate|cocaine|methamphetamine|stimulant use|cannabis use"
                      r"|addict|use disorder|dependence"),
    "Eating disorders": (r"anorexia nervosa|bulimi|binge.?eating|eating disorder|\barfid\b"
                         r"|avoidant.?restrictive|disordered eating|feeding disorder,? (of )?infancy"),
}

def group_of(text):
    t = (text or "").lower()
    if re.search(EXCLUDE, t):
        return "Other"
    for name, pattern in GROUPS.items():
        if re.search(pattern, t):
            return name
    return "Other"

conditions["condition_group"] = conditions["condition"].map(group_of)
print(conditions["condition_group"].value_counts())

condition_group
Other                        19797
Depression                    7893
Anxiety                       4661
Schizophrenia & psychosis     3021
PTSD                          1801
Substance use                 1307
Eating disorders               924
Bipolar disorder               783
OCD                            449
Name: count, dtype: int64


In [61]:
PSYCHEDELIC = (
    # substances
    r"psilocybin|psilocin|mdma|midomafetamine|\blsd\b|lysergic"
    r"|\bdmt\b|dimethyltryptamine|5-meo|ayahuasca|ibogaine|noribogaine"
    r"|mescaline|salvinorin|psychedelic"
    # company drug codes
    r"|comp ?360"        # Compass Pathways, psilocybin
    r"|mm-?120"          # MindMed, LSD
    r"|cyb00[34]"        # Cybin, deuterated psilocybin analogue / DMT
    r"|gh00[12]"         # GH Research, 5-MeO-DMT
    r"|bpl-?003"         # Beckley Psytech, 5-MeO-DMT nasal spray
    r"|spl0?26"          # Small Pharma, DMT
    r"|re-?104"          # Reunion Neuroscience, psilocybin-like prodrug
    r"|bmnd0\d"          # Biomind Labs, 5-MeO-DMT
)
KETAMINE = r"ketamine|spravato"   # also matches esketamine and arketamine

def drug_class(name):
    n = (name or "").lower()
    if n.startswith("placebo") or "matched placebo" in n:
        return None
    if re.search(PSYCHEDELIC, n):
        return "Psychedelic / MDMA"
    if re.search(KETAMINE, n):
        return "Ketamine / esketamine"
    return None

interventions["psychedelic_class"] = interventions["intervention_name"].map(drug_class)
print(interventions["psychedelic_class"].value_counts())

psychedelic_class
Ketamine / esketamine    415
Psychedelic / MDMA       203
Name: count, dtype: int64


In [62]:
grouped_ids = set(conditions.loc[conditions.condition_group != "Other", "nct_id"])
trials["in_scope"] = trials["nct_id"].isin(grouped_ids).astype(int)
print(trials["in_scope"].value_counts())

flagged = interventions.dropna(subset=["psychedelic_class"])
summary = (flagged.assign(in_scope=flagged.nct_id.isin(grouped_ids))
                  .groupby(["psychedelic_class", "in_scope"])["nct_id"].nunique()
                  .unstack(fill_value=0))
print(summary)

in_scope
1    14713
0     3644
Name: count, dtype: int64
in_scope               False  True 
psychedelic_class                  
Ketamine / esketamine     25    318
Psychedelic / MDMA         7    165


In [63]:
import sqlite3

tables = {"trials": trials, "conditions": conditions,
          "interventions": interventions, "sites": sites}
for name, df in tables.items():
    df.to_csv(f"{PROJECT}/data/clean/{name}.csv", index=False)

con = sqlite3.connect(f"{PROJECT}/data/trials.db")
for name, df in tables.items():
    df.to_sql(name, con, if_exists="replace", index=False)

def q(sql):
    return pd.read_sql(sql, con)

print(q("SELECT name FROM sqlite_master WHERE type = 'table'"))

            name
0         trials
1     conditions
2  interventions
3          sites


In [ ]:
d1 = q("""
SELECT t.start_year, c.condition_group,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t
JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY t.start_year, c.condition_group
ORDER BY t.start_year, c.condition_group;
""")
print(d1.pivot(index="start_year", columns="condition_group", values="trials").fillna(0).astype(int).to_string())

condition_group  Anxiety  Bipolar disorder  Depression  Eating disorders  OCD  PTSD  Schizophrenia & psychosis  Substance use
start_year                                                                                                                   
2010                  80                50         265                17   11    71                        127             23
2011                  85                40         267                15   20    62                        136             26
2012                 102                40         273                24   20    86                        136             32
2013                  99                42         244                20   19    74                        128             35
2014                 141                54         304                24   29    67                        155             29
2015                 163                36         326                36   23    77                        125        

In [ ]:
print(q("""
SELECT t.start_year,
       COUNT(DISTINCT t.nct_id) AS anxiety_trials,
       COUNT(DISTINCT CASE WHEN LOWER(t.title) LIKE '%covid%'
                            OR LOWER(t.title) LIKE '%pandemic%'
                            OR LOWER(t.title) LIKE '%coronavirus%'
                            OR LOWER(t.title) LIKE '%sars-cov%'
                           THEN t.nct_id END) AS covid_in_title
FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group = 'Anxiety' AND t.start_year BETWEEN 2018 AND 2023
GROUP BY t.start_year ORDER BY t.start_year;
""").to_string())

   start_year  anxiety_trials  covid_in_title
0        2018             239               0
1        2019             258               0
2        2020             263              38
3        2021             415              25
4        2022             392               7
5        2023             477               7


In [ ]:
d2 = q("""
WITH yearly AS (
  SELECT t.start_year, c.condition_group, COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN conditions c ON t.nct_id = c.nct_id
  WHERE c.condition_group <> 'Other'
  GROUP BY t.start_year, c.condition_group
)
SELECT start_year, condition_group, trials,
       LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year) AS prev_year,
       ROUND(100.0 * (trials - LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year))
             / LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year), 1) AS pct_change
FROM yearly
ORDER BY condition_group, start_year;
""")
print(d2.pivot(index="start_year", columns="condition_group", values="pct_change").to_string())

condition_group  Anxiety  Bipolar disorder  Depression  Eating disorders   OCD  PTSD  Schizophrenia & psychosis  Substance use
start_year                                                                                                                    
2010                 NaN               NaN         NaN               NaN   NaN   NaN                        NaN            NaN
2011                 6.3             -20.0         0.8             -11.8  81.8 -12.7                        7.1           13.0
2012                20.0               0.0         2.2              60.0   0.0  38.7                        0.0           23.1
2013                -2.9               5.0       -10.6             -16.7  -5.0 -14.0                       -5.9            9.4
2014                42.4              28.6        24.6              20.0  52.6  -9.5                       21.1          -17.1
2015                15.6             -33.3         7.2              50.0 -20.7  14.9                      -19.4

## Findings so far

**D1: Trials per year by condition (2010–12 avg → 2023–25 avg)**
- Anxiety ×5.7, substance use ×3.5, eating disorders ×3.5, depression ×2.5, OCD and PTSD ×2.0
- Schizophrenia and bipolar disorder are nearly flat (×1.2): research on common disorders grew far faster than research on serious mental illness
- Registry growth inflates all counts, so compare groups with each other rather than citing raw growth
- No visible drop in 2024–25, so reporting lag isn't a problem

**COVID check (anxiety trials with COVID in the title)**
- 2020: 38 of 263 anxiety trials were COVID-related; non-COVID anxiety trials fell from 258 to 225
- 2021: anxiety rose to 415, but only 25 were COVID-related; non-COVID trials stepped up to ~390 and stayed there
- The 2021 rise is not explained by COVID-specific trials

**D2: Year-over-year change (LAG)**
- 2020: 7 of 8 groups fell (anxiety held only because of COVID trials)
- 2021: all 8 groups rose
- OCD, eating disorders and bipolar are too small for year-to-year % to mean much

In [ ]:
d3 = q("""
WITH typed AS (
  SELECT DISTINCT t.nct_id, t.start_year, i.intervention_type
  FROM trials t
  JOIN interventions i ON t.nct_id = i.nct_id
  WHERE t.in_scope = 1
)
SELECT start_year, intervention_type, COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY start_year), 1) AS pct_of_year
FROM typed
GROUP BY start_year, intervention_type
ORDER BY start_year, trials DESC;
""")
print(d3.pivot(index="start_year", columns="intervention_type", values="pct_of_year").fillna(0).to_string())

intervention_type  BEHAVIORAL  BIOLOGICAL  COMBINATION_PRODUCT  DEVICE  DIAGNOSTIC_TEST  DIETARY_SUPPLEMENT  DRUG  GENETIC  OTHER  PROCEDURE  RADIATION
start_year                                                                                                                                             
2010                     32.5         0.2                  0.0     4.3              0.0                 2.1  44.3      0.3   13.6        2.1        0.6
2011                     34.0         0.2                  0.0     7.2              0.0                 1.7  40.5      0.0   13.8        2.7        0.0
2012                     38.5         0.5                  0.0     6.2              0.0                 1.4  36.2      0.0   13.7        3.6        0.0
2013                     38.3         0.2                  0.0    10.2              0.0                 1.9  31.5      0.0   16.2        1.9        0.0
2014                     38.8         0.8                  0.0     8.8              0.1 

In [ ]:
# 1. Actual trial counts by type, for a few sample years
print(d3.pivot(index="start_year", columns="intervention_type", values="trials")
        .fillna(0).astype(int)[["DRUG", "BEHAVIORAL", "DEVICE", "OTHER"]]
        .loc[[2010, 2015, 2020, 2025]].to_string())

# 2. What's inside OTHER and DEVICE in recent years
for kind in ["OTHER", "DEVICE"]:
    print(f"\n{kind}, 2023–2025:")
    print(q(f"""
    SELECT i.intervention_name, COUNT(DISTINCT t.nct_id) AS trials
    FROM trials t JOIN interventions i ON t.nct_id = i.nct_id
    WHERE t.in_scope = 1 AND t.start_year >= 2023 AND i.intervention_type = '{kind}'
    GROUP BY i.intervention_name ORDER BY trials DESC LIMIT 15;
    """).to_string())

intervention_type  DRUG  BEHAVIORAL  DEVICE  OTHER
start_year                                        
2010                277         203      27     85
2015                211         337      86    144
2020                173         442     120    223
2025                251         688     272    356

OTHER, 2023–2025:
           intervention_name  trials
0                    Placebo      62
1                    Control      12
2   Treatment as Usual (TAU)       9
3         Treatment as usual       8
4            Psychoeducation       8
5               routine care       7
6            No intervention       7
7                   Exercise       7
8   Treatment as usual (TAU)       6
9                   Waitlist       5
10           Virtual Reality       5
11        Music intervention       5
12                     Music       5
13          Mandala coloring       5
14             control group       4

DEVICE, 2023–2025:
                                      intervention_name  trials

**D3: Treatment types (in-scope trials)**
- Drug share fell from 44% (2010) to 15% (2025), but drug trial counts stayed roughly level (277 → 251); non-drug treatments grew around them
- Behavioural trials grew ~3.4× (203 → 691) and are the largest type every year
- Device trials grew ~10× (27 → 272), driven by brain stimulation (TMS, rTMS, iTBS, tDCS) and virtual reality; by 2024–25 device trials matched drug trials in share
- "Other" is mostly comparison arms (placebo, treatment as usual, waitlist) plus non-drug therapies labelled inconsistently, so it's excluded from the treatment-type chart and the behavioural count is likely an undercount

In [ ]:
d4 = q("""
WITH counts AS (
  SELECT c.condition_group, t.sponsor_name, t.sponsor_class,
         COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN conditions c ON t.nct_id = c.nct_id
  WHERE c.condition_group <> 'Other'
  GROUP BY c.condition_group, t.sponsor_name, t.sponsor_class
),
ranked AS (
  SELECT *, RANK() OVER (PARTITION BY condition_group ORDER BY trials DESC) AS rnk
  FROM counts
)
SELECT * FROM ranked WHERE rnk <= 5
ORDER BY condition_group, rnk;
""")
print(d4.to_string())

              condition_group                                                  sponsor_name sponsor_class  trials  rnk
0                     Anxiety                                         Karolinska Institutet         OTHER      54    1
1                     Anxiety                                Massachusetts General Hospital         OTHER      49    2
2                     Anxiety                         University of California, Los Angeles         OTHER      41    3
3                     Anxiety                         VA Office of Research and Development           FED      35    4
4                     Anxiety                                           Stanford University         OTHER      34    5
5            Bipolar disorder                                Massachusetts General Hospital         OTHER      29    1
6            Bipolar disorder                                               Mclean Hospital         OTHER      18    2
7            Bipolar disorder                   

**D4: Top 5 lead sponsors per condition**
- 40 of 41 top-5 slots are academic, hospital or government; the only company is Otsuka (#1 in schizophrenia, 60 trials)
- The US VA dominates PTSD (185 trials, ~6× the next sponsor) and also leads substance use
- CAMH (Toronto) is top 5 in four groups: schizophrenia #2, substance use #4, depression #5, bipolar #5
- Massachusetts General Hospital leads depression (126) and appears in 5 of 8 groups
- Caveat: lead sponsors only; industry is spread across many companies, so per-sponsor rankings understate it

In [ ]:
d4b = q("""
SELECT t.start_year, t.sponsor_class, COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY t.start_year), 1) AS pct_of_year
FROM trials t
WHERE t.in_scope = 1
GROUP BY t.start_year, t.sponsor_class
ORDER BY t.start_year, trials DESC;
""")
print(d4b.pivot(index="start_year", columns="sponsor_class", values="pct_of_year").fillna(0).to_string())

sponsor_class  FED  INDIV  INDUSTRY  NETWORK  NIH  OTHER  OTHER_GOV  UNKNOWN
start_year                                                                  
2010           5.6    0.2      24.2      0.2  0.9   66.1        2.9      0.0
2011           6.0    0.0      21.9      0.2  0.7   68.8        2.4      0.0
2012           4.7    0.0      15.7      0.3  0.7   74.6        4.0      0.0
2013           4.7    0.0      12.6      0.0  0.0   79.4        3.3      0.0
2014           4.4    0.0      11.5      0.1  0.3   81.9        1.8      0.0
2015           6.0    0.1       9.8      0.0  0.3   81.2        2.6      0.0
2016           4.1    0.0      10.7      0.0  0.1   83.1        2.0      0.0
2017           2.5    0.1       9.1      0.1  0.3   85.0        2.9      0.0
2018           3.7    0.1      11.3      0.1  0.4   82.0        2.4      0.0
2019           4.1    0.0       8.1      0.1  0.2   85.4        2.1      0.0
2020           2.2    0.0       9.5      0.0  0.1   85.3        2.9      0.0

In [ ]:
print(d4b.pivot(index="start_year", columns="sponsor_class", values="trials")
         .fillna(0).astype(int)[["INDUSTRY", "OTHER", "FED"]]
         .loc[[2010, 2015, 2020, 2025]].to_string())

sponsor_class  INDUSTRY  OTHER  FED
start_year                         
2010                134    366   31
2015                 69    570   42
2020                 85    766   20
2025                126   1265   18


**D4b: Sponsor type over time (in-scope trials)**
- Industry share fell from 24% (2010) to 9% (2025); most of the drop came by 2015, then flat at ~8–11%
- Universities and hospitals rose from 66% to 87%
- US federal (FED) share fell from 6% to 1%
- Fits D3: the fastest-growing treatment types (behavioural, device) are mostly academic
- - Counts: industry trials halved from 2010 to 2015 (134 → 69), then recovered to 126 by 2025; academic trials grew ~3.5× (366 → 1,269), so industry's *share* kept falling even as its numbers recovered
- The industry dip and recovery mirrors drug trials in D3; check whether D7 (psychedelics/ketamine) helps explain the recovery

In [ ]:
d5 = q("""
SELECT sponsor_class,
       COUNT(*) AS ended_trials,
       SUM(overall_status = 'COMPLETED') AS completed,
       SUM(overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')) AS stopped_early,
       ROUND(100.0 * SUM(overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED'))
             / COUNT(*), 1) AS pct_stopped_early
FROM trials
WHERE in_scope = 1
  AND overall_status IN ('COMPLETED','TERMINATED','WITHDRAWN','SUSPENDED')
GROUP BY sponsor_class
ORDER BY ended_trials DESC;
""")
print(d5.to_string())

print(q("""
SELECT why_stopped, COUNT(*) AS trials
FROM trials
WHERE in_scope = 1 AND why_stopped IS NOT NULL
GROUP BY why_stopped ORDER BY trials DESC LIMIT 20;
""").to_string())

  sponsor_class  ended_trials  completed  stopped_early  pct_stopped_early
0         OTHER          8025       7038            987               12.3
1      INDUSTRY          1336       1123            213               15.9
2           FED           374        326             48               12.8
3     OTHER_GOV           179        169             10                5.6
4           NIH            32         22             10               31.3
5       NETWORK             8          6              2               25.0
6         INDIV             4          2              2               50.0
                                                                                                                                                      why_stopped  trials
0                                                                                                                                                        COVID-19      14
1                                                           

**D5: Completed vs stopped early (ended trials)**
- Industry stops early most often: 15.9% vs 12.3% for universities/hospitals and 12.8% for US federal
- NIH, network and individual sponsors are too small to compare (<35 ended trials each)
- why_stopped is free text with many spellings of the same reason, plus blank strings; grouped into categories with CASE WHEN (D5b)

In [ ]:
d5b = q("""
WITH reasons AS (
  SELECT sponsor_class, LOWER(why_stopped) AS r
  FROM trials
  WHERE in_scope = 1
    AND overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')
    AND TRIM(COALESCE(why_stopped, '')) <> ''
),
grouped AS (
  SELECT sponsor_class,
    CASE
      WHEN r LIKE '%covid%' OR r LIKE '%pandemic%' OR r LIKE '%coronavirus%' THEN 'COVID-19'
      WHEN r LIKE '%efficacy%' OR r LIKE '%futility%' OR r LIKE '%interim analys%' THEN 'Efficacy / futility'
      WHEN r LIKE '%sponsor decision%' OR r LIKE '%business%' OR r LIKE '%strategic%'
        OR r LIKE '%portfolio%' OR r LIKE '%company decision%' OR r LIKE '%program%' THEN 'Sponsor / business decision'
      WHEN r LIKE '%fund%' OR r LIKE '%budget%' OR r LIKE '%financ%' THEN 'Funding'
      WHEN r LIKE '%recruit%' OR r LIKE '%enrol%' OR r LIKE '%accrual%'
        OR r LIKE '%participant%' OR r LIKE '%subject%' OR r LIKE '%patient%' THEN 'Recruitment'
      WHEN r LIKE '%pi %' OR r LIKE '%investigator%' OR r LIKE '%left the%'
        OR r LIKE '%staff%' OR r LIKE '%personnel%' THEN 'Investigator / staff'
      WHEN r LIKE '%safety%' OR r LIKE '%adverse%' THEN 'Safety'
      ELSE 'Other / unclear'
    END AS reason_group
  FROM reasons
)
SELECT reason_group,
       COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
       SUM(sponsor_class = 'INDUSTRY') AS industry,
       SUM(sponsor_class = 'OTHER') AS academic
FROM grouped
GROUP BY reason_group
ORDER BY trials DESC;
""")
print(d5b.to_string())

                  reason_group  trials   pct  industry  academic
0                  Recruitment     331  27.3        41       272
1              Other / unclear     306  25.3        47       245
2                      Funding     197  16.3        11       178
3                     COVID-19     146  12.1         5       135
4         Investigator / staff     112   9.2         1        95
5  Sponsor / business decision      62   5.1        54         7
6          Efficacy / futility      46   3.8        33        13
7                       Safety      11   0.9         6         4


**D5b: Why trials stop (1,211 stopped trials with a reason, grouped with CASE WHEN)**
- Overall: recruitment 27%, funding 16%, COVID 12%, investigator/staff 9%, sponsor decision 5%, efficacy/futility 4%, safety <1%
- Industry stops for strategy or results: business decision 27%, recruitment 21%, efficacy/futility 17%
- Academia stops for resources: recruitment 29%, funding 19%, COVID 14%, investigator/staff 10%
- Safety is almost never the stated reason (11 trials)
- ~25% of reasons are uncategorised; checked: a long tail of one-off wordings (none used more than twice), so no category is missing

In [ ]:
print(q("""
SELECT LOWER(why_stopped) AS r, COUNT(*) AS n
FROM trials
WHERE in_scope = 1 AND overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')
  AND TRIM(COALESCE(why_stopped,'')) <> ''
GROUP BY r ORDER BY n DESC LIMIT 25;
""").to_string())

                                                                                                                                                                r   n
0                                                                                                                                                        covid-19  17
1                                                                                                                                                 lack of funding  15
2                                                                                                                                        recruitment difficulties   8
3                                                                                                                                             pi left institution   8
4                                                                                                                                        no participants enrolled   8
5   

In [ ]:
print(q("""
WITH reasons AS (
  SELECT LOWER(why_stopped) AS r
  FROM trials
  WHERE in_scope = 1
    AND overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')
    AND TRIM(COALESCE(why_stopped, '')) <> ''
)
SELECT r, COUNT(*) AS n
FROM reasons
WHERE NOT (r LIKE '%covid%' OR r LIKE '%pandemic%' OR r LIKE '%coronavirus%'
        OR r LIKE '%efficacy%' OR r LIKE '%futility%' OR r LIKE '%interim analys%'
        OR r LIKE '%sponsor decision%' OR r LIKE '%business%' OR r LIKE '%strategic%'
        OR r LIKE '%portfolio%' OR r LIKE '%company decision%' OR r LIKE '%program%'
        OR r LIKE '%fund%' OR r LIKE '%budget%' OR r LIKE '%financ%'
        OR r LIKE '%recruit%' OR r LIKE '%enrol%' OR r LIKE '%accrual%'
        OR r LIKE '%participant%' OR r LIKE '%subject%' OR r LIKE '%patient%'
        OR r LIKE '%pi %' OR r LIKE '%investigator%' OR r LIKE '%left the%'
        OR r LIKE '%staff%' OR r LIKE '%personnel%'
        OR r LIKE '%safety%' OR r LIKE '%adverse%')
GROUP BY r ORDER BY n DESC LIMIT 25;
""").to_string())

                                                                                                                                                                                                                                 r  n
0                                                                                                                                                                                                       toxic epidermal necrolysis  2
1                                                                                                                                                            the trial has been temporarily suspended due to administrative issues  2
2                                                                                                                                                                                                                         terminée  2
3   suspended (the ut irb determined that cbd oil is being used as a drug as def

In [ ]:
d6 = q("""
SELECT c.condition_group,
       COUNT(DISTINCT c.nct_id) AS all_trials,
       COUNT(DISTINCT CASE WHEN s.country = 'Canada' THEN c.nct_id END) AS with_canadian_site,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN s.country = 'Canada' THEN c.nct_id END)
             / COUNT(DISTINCT c.nct_id), 1) AS pct_canada
FROM conditions c
LEFT JOIN sites s ON c.nct_id = s.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY c.condition_group
ORDER BY pct_canada DESC;
""")
print(d6.to_string())

print(q("""
SELECT COUNT(DISTINCT t.nct_id) AS trials_with_canadian_site
FROM trials t JOIN sites s ON t.nct_id = s.nct_id
WHERE t.in_scope = 1 AND s.country = 'Canada';
"""))

             condition_group  all_trials  with_canadian_site  pct_canada
0                        OCD         420                  36         8.6
1           Bipolar disorder         689                  58         8.4
2                 Depression        6936                 568         8.2
3  Schizophrenia & psychosis        2243                 144         6.4
4           Eating disorders         649                  41         6.3
5                    Anxiety        4140                 255         6.2
6              Substance use         955                  58         6.1
7                       PTSD        1674                  88         5.3
   trials_with_canadian_site
0                       1019


In [ ]:
# 1. How many in-scope trials list no site at all? (They count in the denominator above.)
print(q("""
SELECT COUNT(*) AS in_scope_trials,
       SUM(nct_id NOT IN (SELECT nct_id FROM sites)) AS no_site_listed
FROM trials WHERE in_scope = 1;
""").to_string())

# 2. Where does Canada rank among countries?
print(q("""
SELECT s.country, COUNT(DISTINCT t.nct_id) AS trials,
       ROUND(100.0 * COUNT(DISTINCT t.nct_id) / (SELECT COUNT(*) FROM trials WHERE in_scope = 1), 1) AS pct
FROM trials t JOIN sites s ON t.nct_id = s.nct_id
WHERE t.in_scope = 1
GROUP BY s.country ORDER BY trials DESC LIMIT 10;
""").to_string())

   in_scope_trials  no_site_listed
0            14700             908
            country  trials   pct
0     United States    6429  43.7
1            Canada    1019   6.9
2  Turkey (Türkiye)     811   5.5
3             China     733   5.0
4            France     632   4.3
5    United Kingdom     500   3.4
6           Germany     464   3.2
7             Spain     418   2.8
8            Sweden     339   2.3
9            Taiwan     284   1.9


**D6: Canada (Question 5)**
- 1,020 of 14,713 in-scope trials (6.9%) have at least one Canadian site; 7.4% of trials that list any site
- Canada ranks #2 in ClinicalTrials.gov after the US (6,429; 43.7%), ahead of Turkey, China, France and the UK
- Per person, roughly 25 trials per million in Canada vs ~19 in the US
- Canadian share is fairly even across groups: highest in OCD (8.6%), bipolar (8.4%) and depression (8.2%), lowest in PTSD (5.3%, where the US VA dominates)
- Caveat: ClinicalTrials.gov is US-based; many European and Chinese trials register elsewhere, so country ranks reflect this registry
- 908 in-scope trials (6.2%) list no site, so percentages are slight underestimates

In [ ]:
d7 = q("""
WITH yearly AS (
  SELECT t.start_year, i.psychedelic_class, COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN interventions i ON t.nct_id = i.nct_id
  WHERE i.psychedelic_class IS NOT NULL AND t.in_scope = 1
  GROUP BY t.start_year, i.psychedelic_class
)
SELECT start_year, psychedelic_class, trials,
       SUM(trials) OVER (PARTITION BY psychedelic_class ORDER BY start_year) AS running_total
FROM yearly
ORDER BY psychedelic_class, start_year;
""")
print(d7.pivot(index="start_year", columns="psychedelic_class", values="trials").fillna(0).astype(int).to_string())
print(d7.pivot(index="start_year", columns="psychedelic_class", values="running_total").ffill().fillna(0).astype(int).to_string())

psychedelic_class  Ketamine / esketamine  Psychedelic / MDMA
start_year                                                  
2010                                   8                   2
2011                                   4                   0
2012                                  16                   1
2013                                  15                   2
2014                                  17                   3
2015                                  30                   1
2016                                  26                   2
2017                                  19                   3
2018                                  16                   7
2019                                  22                   7
2020                                  20                   5
2021                                  26                  19
2022                                  27                  16
2023                                  19                  29
2024                    

**D7: Psychedelic and ketamine trials (in scope; running totals 318 ketamine, 165 psychedelic)**
- Psychedelic trials: 0–3 a year until 2017, 5–7 in 2018–20, then a surge from 2021 (19, 16, 29, 37, 31)
- 80% of psychedelic trials (132 of 165) started in 2021 or later
- Psychedelic trials outnumbered ketamine trials in 2023, 2024 and 2025
- Ketamine is steady at about 15–30 trials a year since 2012; no visible jump after esketamine's 2019 approval
- The 2021 surge was academic-led (university trials 17 → 95); industry's share of psychedelic trials fell from 48% to 23%
- Industry share of ketamine trials fell from 17% to 7% after esketamine's 2019 approval
- Industry is overrepresented in psychedelic research: it sponsors 23% of 2021–25 psychedelic trials (31 of 132), vs about 9% of all mental health trials; universities and hospitals still sponsor the majority (95)
- Psychedelic trials are too few to explain why industry-sponsored trials recovered overall (from 69 in 2015 to 126 in 2025)

In [ ]:
print(q("""
SELECT i.psychedelic_class,
       CASE WHEN t.start_year >= 2021 THEN '2021-2025' ELSE '2010-2020' END AS period,
       t.sponsor_class,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t JOIN interventions i ON t.nct_id = i.nct_id
WHERE i.psychedelic_class IS NOT NULL AND t.in_scope = 1
GROUP BY 1, 2, 3
ORDER BY 1, 2, trials DESC;
""").to_string())

        psychedelic_class     period sponsor_class  trials
0   Ketamine / esketamine  2010-2020         OTHER     140
1   Ketamine / esketamine  2010-2020      INDUSTRY      32
2   Ketamine / esketamine  2010-2020           FED      14
3   Ketamine / esketamine  2010-2020     OTHER_GOV       5
4   Ketamine / esketamine  2010-2020           NIH       2
5   Ketamine / esketamine  2021-2025         OTHER     108
6   Ketamine / esketamine  2021-2025      INDUSTRY       9
7   Ketamine / esketamine  2021-2025           FED       5
8   Ketamine / esketamine  2021-2025           NIH       2
9   Ketamine / esketamine  2021-2025     OTHER_GOV       1
10     Psychedelic / MDMA  2010-2020         OTHER      17
11     Psychedelic / MDMA  2010-2020      INDUSTRY      16
12     Psychedelic / MDMA  2021-2025         OTHER      95
13     Psychedelic / MDMA  2021-2025      INDUSTRY      31
14     Psychedelic / MDMA  2021-2025     OTHER_GOV       4
15     Psychedelic / MDMA  2021-2025           FED      

In [64]:
import os
os.makedirs(f"{PROJECT}/sql", exist_ok=True)
os.makedirs(f"{PROJECT}/results", exist_ok=True)

QUERIES = {
"01_trials_per_year_by_condition": """
SELECT t.start_year, c.condition_group,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t
JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY t.start_year, c.condition_group
ORDER BY t.start_year, c.condition_group;
""",

"02_year_over_year_change": """
WITH yearly AS (
  SELECT t.start_year, c.condition_group, COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN conditions c ON t.nct_id = c.nct_id
  WHERE c.condition_group <> 'Other'
  GROUP BY t.start_year, c.condition_group
)
SELECT start_year, condition_group, trials,
       LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year) AS prev_year,
       ROUND(100.0 * (trials - LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year))
             / LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year), 1) AS pct_change
FROM yearly
ORDER BY condition_group, start_year;
""",

"03_treatment_type_share": """
WITH typed AS (
  SELECT DISTINCT t.nct_id, t.start_year, i.intervention_type
  FROM trials t
  JOIN interventions i ON t.nct_id = i.nct_id
  WHERE t.in_scope = 1
)
SELECT start_year, intervention_type, COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY start_year), 1) AS pct_of_year
FROM typed
GROUP BY start_year, intervention_type
ORDER BY start_year, trials DESC;
""",

"04a_top_sponsors_by_condition": """
WITH counts AS (
  SELECT c.condition_group, t.sponsor_name, t.sponsor_class,
         COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN conditions c ON t.nct_id = c.nct_id
  WHERE c.condition_group <> 'Other'
  GROUP BY c.condition_group, t.sponsor_name, t.sponsor_class
),
ranked AS (
  SELECT *, RANK() OVER (PARTITION BY condition_group ORDER BY trials DESC) AS rnk
  FROM counts
)
SELECT * FROM ranked WHERE rnk <= 5
ORDER BY condition_group, rnk;
""",

"04b_sponsor_type_by_year": """
SELECT t.start_year, t.sponsor_class, COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY t.start_year), 1) AS pct_of_year
FROM trials t
WHERE t.in_scope = 1
GROUP BY t.start_year, t.sponsor_class
ORDER BY t.start_year, trials DESC;
""",

"05a_stopped_early_by_sponsor": """
SELECT sponsor_class,
       COUNT(*) AS ended_trials,
       SUM(overall_status = 'COMPLETED') AS completed,
       SUM(overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')) AS stopped_early,
       ROUND(100.0 * SUM(overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED'))
             / COUNT(*), 1) AS pct_stopped_early
FROM trials
WHERE in_scope = 1
  AND overall_status IN ('COMPLETED','TERMINATED','WITHDRAWN','SUSPENDED')
GROUP BY sponsor_class
ORDER BY ended_trials DESC;
""",

"05b_why_stopped_grouped": """
WITH reasons AS (
  SELECT sponsor_class, LOWER(why_stopped) AS r
  FROM trials
  WHERE in_scope = 1
    AND overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')
    AND TRIM(COALESCE(why_stopped, '')) <> ''
),
grouped AS (
  SELECT sponsor_class,
    CASE
      WHEN r LIKE '%covid%' OR r LIKE '%pandemic%' OR r LIKE '%coronavirus%' THEN 'COVID-19'
      WHEN r LIKE '%efficacy%' OR r LIKE '%futility%' OR r LIKE '%interim analys%' THEN 'Efficacy / futility'
      WHEN r LIKE '%sponsor decision%' OR r LIKE '%business%' OR r LIKE '%strategic%'
        OR r LIKE '%portfolio%' OR r LIKE '%company decision%' OR r LIKE '%program%' THEN 'Sponsor / business decision'
      WHEN r LIKE '%fund%' OR r LIKE '%budget%' OR r LIKE '%financ%' THEN 'Funding'
      WHEN r LIKE '%recruit%' OR r LIKE '%enrol%' OR r LIKE '%accrual%'
        OR r LIKE '%participant%' OR r LIKE '%subject%' OR r LIKE '%patient%' THEN 'Recruitment'
      WHEN r LIKE '%pi %' OR r LIKE '%investigator%' OR r LIKE '%left the%'
        OR r LIKE '%staff%' OR r LIKE '%personnel%' THEN 'Investigator / staff'
      WHEN r LIKE '%safety%' OR r LIKE '%adverse%' THEN 'Safety'
      ELSE 'Other / unclear'
    END AS reason_group
  FROM reasons
)
SELECT reason_group,
       COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
       SUM(sponsor_class = 'INDUSTRY') AS industry,
       SUM(sponsor_class = 'OTHER') AS academic
FROM grouped
GROUP BY reason_group
ORDER BY trials DESC;
""",

"06a_canadian_sites_by_condition": """
SELECT c.condition_group,
       COUNT(DISTINCT c.nct_id) AS all_trials,
       COUNT(DISTINCT CASE WHEN s.country = 'Canada' THEN c.nct_id END) AS with_canadian_site,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN s.country = 'Canada' THEN c.nct_id END)
             / COUNT(DISTINCT c.nct_id), 1) AS pct_canada
FROM conditions c
LEFT JOIN sites s ON c.nct_id = s.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY c.condition_group
ORDER BY pct_canada DESC;
""",

"06b_top_countries": """
SELECT s.country, COUNT(DISTINCT t.nct_id) AS trials,
       ROUND(100.0 * COUNT(DISTINCT t.nct_id) / (SELECT COUNT(*) FROM trials WHERE in_scope = 1), 1) AS pct
FROM trials t JOIN sites s ON t.nct_id = s.nct_id
WHERE t.in_scope = 1
GROUP BY s.country ORDER BY trials DESC LIMIT 10;
""",

"07a_psychedelic_running_total": """
WITH yearly AS (
  SELECT t.start_year, i.psychedelic_class, COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN interventions i ON t.nct_id = i.nct_id
  WHERE i.psychedelic_class IS NOT NULL AND t.in_scope = 1
  GROUP BY t.start_year, i.psychedelic_class
)
SELECT start_year, psychedelic_class, trials,
       SUM(trials) OVER (PARTITION BY psychedelic_class ORDER BY start_year) AS running_total
FROM yearly
ORDER BY psychedelic_class, start_year;
""",

"07b_psychedelic_sponsors_by_period": """
SELECT i.psychedelic_class,
       CASE WHEN t.start_year >= 2021 THEN '2021-2025' ELSE '2010-2020' END AS period,
       t.sponsor_class,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t JOIN interventions i ON t.nct_id = i.nct_id
WHERE i.psychedelic_class IS NOT NULL AND t.in_scope = 1
GROUP BY 1, 2, 3
ORDER BY 1, 2, trials DESC;
""",
}

for name, sql in QUERIES.items():
    with open(f"{PROJECT}/sql/{name}.sql", "w") as f:
        f.write(sql.strip() + "\n")
    result = q(sql)
    result.to_csv(f"{PROJECT}/results/{name}.csv", index=False)
    print(f"{name}: {len(result)} rows")

01_trials_per_year_by_condition: 128 rows
02_year_over_year_change: 128 rows
03_treatment_type_share: 153 rows
04a_top_sponsors_by_condition: 41 rows
04b_sponsor_type_by_year: 97 rows
05a_stopped_early_by_sponsor: 7 rows
05b_why_stopped_grouped: 8 rows
06a_canadian_sites_by_condition: 8 rows
06b_top_countries: 10 rows
07a_psychedelic_running_total: 31 rows
07b_psychedelic_sponsors_by_period: 16 rows


In [65]:
print(q("""
SELECT c.condition_group, t.start_year, COUNT(DISTINCT t.nct_id) AS my_count
FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE (c.condition_group = 'Schizophrenia & psychosis' AND t.start_year = 2020)
   OR (c.condition_group = 'PTSD' AND t.start_year = 2022)
   OR (c.condition_group = 'Eating disorders' AND t.start_year = 2024)
GROUP BY 1, 2;
""").to_string())

             condition_group  start_year  my_count
0           Eating disorders        2024        70
1                       PTSD        2022       143
2  Schizophrenia & psychosis        2020       121


In [57]:
print(q("""
SELECT c.condition, c.condition_group, COUNT(DISTINCT t.nct_id) AS trials
FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE t.start_year = 2024
  AND (LOWER(c.condition) LIKE '%eat%' OR LOWER(c.condition) LIKE '%feed%'
    OR LOWER(c.condition) LIKE '%food%' OR LOWER(c.condition) LIKE '%obes%'
    OR LOWER(c.condition) LIKE '%weight%' OR LOWER(c.condition) LIKE '%anorex%'
    OR LOWER(c.condition) LIKE '%bulim%' OR LOWER(c.condition) LIKE '%binge%'
    OR LOWER(c.condition) LIKE '%arfid%' OR LOWER(c.condition) LIKE '%body image%')
GROUP BY c.condition, c.condition_group
ORDER BY trials DESC LIMIT 30;
""").to_string())

                                            condition   condition_group  trials
0                      Treatment Resistant Depression        Depression      24
1                                    Eating Disorders  Eating disorders      23
2                                   Suicidal Ideation             Other      17
3                                    Anorexia Nervosa  Eating disorders      16
4                                        Binge Eating  Eating disorders      11
5                                             Obesity             Other      10
6                               Binge-Eating Disorder  Eating disorders       6
7                                          Body Image             Other       4
8                                     Bulimia Nervosa  Eating disorders       4
9            Depressive Disorder, Treatment-Resistant        Depression       4
10                        Disordered Eating Behaviors             Other       4
11                       Feeding and Eat

In [58]:
from google.colab import files
import io
up = files.upload()
web = pd.read_csv(io.BytesIO(list(up.values())[0]))
web_ids = set(web["NCT Number"])

my_ids = set(q("""
SELECT DISTINCT t.nct_id FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group = 'Eating disorders' AND t.start_year = 2024
""")["nct_id"])

not_downloaded = web_ids - set(trials["nct_id"])
downloaded_not_grouped = (web_ids & set(trials["nct_id"])) - my_ids
print("On website:", len(web_ids), "| matched:", len(web_ids & my_ids),
      "| not in my download:", len(not_downloaded),
      "| downloaded but not grouped:", len(downloaded_not_grouped))

print(conditions[conditions.nct_id.isin(downloaded_not_grouped)]
      .groupby("condition")["nct_id"].nunique().sort_values(ascending=False).head(25).to_string())

Saving ctgov_eating_disorder_2024.csv.csv to ctgov_eating_disorder_2024.csv.csv
On website: 154 | matched: 62 | not in my download: 67 | downloaded but not grouped: 25
condition
Depression                                      3
Rumination                                      3
Bipolar Disorder                                2
Depressive Symptoms                             2
Disordered Eating Behaviors                     2
Avoidant Restrictive Food Intake Disorder       2
Anxiety                                         2
Obesity                                         2
Time Restricted Eating                          2
Female Athlete Triad                            2
Control                                         1
Children With Medical Complexity                1
Body Image                                      1
Affective Symptoms                              1
Cognitive Avoidance                             1
Depressive Disorder, Major                      1
Diet, Mediterranean   

In [59]:
missing = web[web["NCT Number"].isin(not_downloaded)]
print(missing["Conditions"].str.split("|").explode().str.strip()
      .value_counts().head(30).to_string())

Conditions
Obesity                   21
Time Restricted Eating     8
Adolescent Obesity         5
Overweight and Obesity     5
Body Weight                3
Metabolic Syndrome         3
Overweight                 3
Appetite                   2
Physical Inactivity        2
Time-restricted Eating     2
Sleep                      2
Adolescent Overweight      2
Intermittent Fasting       2
Lifestyle Factors          2
Healthy Lifestyle          2
Hunger                     2
Type 2 Diabetes            2
Eating Behaviors           2
Insulin Resistance         2
Childhood Obesity          2
Malnutrition               2
Weight Loss                2
Adiposity                  2
Sarcopenia                 2
Behavior, Eating           2
Obesity and Overweight     2
Huntington Disease         1
Postmenopausal             1
Obesity & Overweight       1
Fat Disorder               1


In [66]:
# Website match after the fix (uses the CSV you uploaded earlier)
my_ids = set(q("""
SELECT DISTINCT t.nct_id FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group = 'Eating disorders' AND t.start_year = 2024
""")["nct_id"])
print("Website eating disorder trials now matched:", len(web_ids & my_ids), "of", len(web_ids))

# Eating disorders growth
d1 = pd.read_csv(f"{PROJECT}/results/01_trials_per_year_by_condition.csv")
e = d1[d1.condition_group == "Eating disorders"].set_index("start_year")["trials"]
print("Eating disorders 2010-12 avg:", round(e.loc[2010:2012].mean()), "| 2023-25 avg:", round(e.loc[2023:2025].mean()))

# Treatment types and sponsor types (counts)
d3 = pd.read_csv(f"{PROJECT}/results/03_treatment_type_share.csv")
print(d3.pivot(index="start_year", columns="intervention_type", values="trials")
        .loc[[2010, 2015, 2020, 2025], ["DRUG", "BEHAVIORAL", "DEVICE"]].to_string())
d4b = pd.read_csv(f"{PROJECT}/results/04b_sponsor_type_by_year.csv")
print(d4b.pivot(index="start_year", columns="sponsor_class", values="trials")
         .loc[[2010, 2015, 2020, 2025], ["INDUSTRY", "OTHER"]].to_string())

# Stop rates
print(pd.read_csv(f"{PROJECT}/results/05a_stopped_early_by_sponsor.csv").head(3).to_string())

# Canada
print(pd.read_csv(f"{PROJECT}/results/06a_canadian_sites_by_condition.csv").to_string())
print(q("""SELECT COUNT(DISTINCT t.nct_id) AS canada FROM trials t JOIN sites s ON t.nct_id = s.nct_id
           WHERE t.in_scope = 1 AND s.country = 'Canada'"""))

Website eating disorder trials now matched: 67 of 154
Eating disorders 2010-12 avg: 19 | 2023-25 avg: 67
intervention_type   DRUG  BEHAVIORAL  DEVICE
start_year                                  
2010               277.0       203.0    27.0
2015               211.0       337.0    86.0
2020               173.0       443.0   120.0
2025               251.0       691.0   272.0
sponsor_class  INDUSTRY   OTHER
start_year                     
2010              134.0   366.0
2015               69.0   570.0
2020               85.0   767.0
2025              126.0  1269.0
  sponsor_class  ended_trials  completed  stopped_early  pct_stopped_early
0         OTHER          8032       7044            988               12.3
1      INDUSTRY          1336       1123            213               15.9
2           FED           374        326             48               12.8
             condition_group  all_trials  with_canadian_site  pct_canada
0                        OCD         420                  36